# Teammate #1 — Load and prepare the data

Start with the loading cell below. Then create `X`, `y`, and the data-splitting function.
The `...` inside each function is only a placeholder: replace it with your implementation.
This notebook becomes a working experiment once your team fills in the functions.

For this first experiment:

- Predict benign versus malignant. Use **1 = malignant** and **0 = benign**.
  Inspect `data.target_names`: the original scikit-learn labels use the opposite encoding.
- Use these four columns, in this order: `mean radius`, `mean texture`,
  `mean smoothness`, `mean concavity`.
- Treat radius as already available. The model may read **one** additional feature.
  This is an artificial feature budget; these descriptors are not separate clinical tests.

**Your handoff:** the four-column DataFrame `X`, the aligned label array `y`,
and a working `prepare_split` function. Agree on these formats before others start.


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Provided: load the dataset, including column names and label descriptions.
data = load_breast_cancer(as_frame=True)
print('Available features:', list(data.data.columns))
print('Original target labels:', dict(enumerate(data.target_names)))

# TODO: create FEATURES, a list of the four feature names specified above.
# TODO: create X, a DataFrame containing those columns in that order.
# TODO: create y, a NumPy array with 1 for malignant and 0 for benign.
# TODO: inspect X and y: check shapes, missing values, and label counts.
# TODO: choose and record five fixed random seeds before running comparisons.


Available features: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']
Original target labels: {0: 'malignant', 1: 'benign'}


In [ ]:
# Split patients and convert features into above/below-training-median values.
# Inputs:
#   X: the four-column feature DataFrame; row order matches y.
#   y: a 1D NumPy array of diagnosis labels (1 = malignant).
#   seed: an integer controlling reproducible, stratified patient splits.
# Outputs:
#   A dictionary with train_ids, val_ids, test_ids (integer row positions),
#   thresholds (a Series of training medians), and z_train, z_val, z_test.
#   Each z is a DataFrame with the same four columns: 0 = at/below median;
#   1 = above median. Reset each z row index so rows match y[corresponding_ids].
def prepare_split(X, y, seed):
    ...  # TODO: implement this function.


Use approximately **60% training / 20% validation / 20% test** patients.
Keep diagnosis proportions similar using stratification. Compute medians using
training patients only, then apply those same medians to validation and test patients.

Before handing this off, show that the three patient sets do not overlap and
explain which set trains predictions, chooses strategies, and evaluates the final result.


# Teammate #2 — Build the measurement-choice rules

A **strategy** is just a rule for choosing which extra feature to read.
For example:

- If radius is at or below its training median, read **texture**.
- If radius is above its training median, read **concavity**.
- Use radius and the chosen feature to predict benign or malignant.

Another strategy could always read texture, regardless of radius.

There are three possible extra features. Choose one for the lower-radius branch
and one for the higher-radius branch: **3 × 3 = 9 strategies** to compare.
We specify the allowed choices; validation data will select which strategy to use.

To avoid confusing number codes, represent each strategy using a dictionary with
two keys: `at_or_below` and `above`. Each value is the name of the extra feature to read.
For the example above, these values would be `mean texture` and `mean concavity`.

For each strategy, learn the final predictions from training patients. The four
possible leaves are combinations of radius bin (0/1) and acquired-feature bin (0/1).
Use the majority diagnosis at each leaf. If a leaf has no training patients, use
the overall training majority. For a tie, use 1; keep this convention fixed.

**Your handoff:** functions to train a strategy, make predictions, and draw the tree.
The predictor must only use radius and the one feature selected for that patient.


In [ ]:
# Return the most common diagnosis label, using a fallback for an empty group.
# Inputs:
#   labels: a 1D array of 0/1 labels, possibly empty.
#   fallback: the label to use if labels is empty.
# Outputs:
#   One integer label, 0 or 1; use 1 if the nonempty group is tied.
def majority(labels, fallback):
    ...  # TODO: implement this function.


In [ ]:
# Learn the four final predictions for one measurement-choice strategy.
# Inputs:
#   z_train: Teammate #1's binarized training DataFrame.
#   y_train: the matching training labels in the same row order.
#   choices: a dictionary: at_or_below and above each name an extra feature.
# Outputs:
#   A model dictionary containing choices, leaves, and counts.
#   leaves maps (radius_bin, acquired_feature_bin) to a learned label.
#   counts maps those same four keys to their training-patient counts.
def fit_policy(z_train, y_train, choices):
    ...  # TODO: implement this function.


In [ ]:
# Follow a trained strategy and predict without reading unchosen features.
# Inputs:
#   model: a model returned by fit_policy.
#   z: a binarized DataFrame with the same four feature columns.
# Outputs:
#   Two aligned NumPy arrays: predicted 0/1 labels and chosen feature names.
#   Each patient must use radius and exactly one extra feature.
def predict_policy(model, z):
    ...  # TODO: implement this function.


In [ ]:
# Draw the measurement choices and final predictions as a small tree.
# Inputs:
#   model: a model returned by fit_policy.
#   thresholds: the training-median Series returned by prepare_split.
# Outputs:
#   A matplotlib Figure showing the radius split, extra feature on each
#   branch, and the prediction and training count at each leaf.
def draw_policy(model, thresholds):
    ...  # TODO: implement this function.


Check one patient by hand: which radius branch do they follow, which extra
feature is read, and which leaf prediction is returned?

Also change that patient's **unchosen** features. The chosen feature and prediction
must stay the same. Work with Teammate #3 to check this across all test patients.


# Teammate #3 — Compare strategies fairly

Use Teammate #1's splits and Teammate #2's prediction functions.

Compare five approaches:

1. **Initial only:** predict using radius alone.
2. **Random extra:** randomly choose one extra feature per patient.
3. **Best fixed extra:** always read the same extra feature; validation chooses which one.
4. **Adaptive extra:** validation chooses among all nine strategies from Teammate #2.
5. **All four:** use every selected feature as a reference with a larger budget.

For the initial-only and all-four references, predict the majority training label
for each observed combination of binary values. Use the same fallback/tie rule as
Teammate #2. This keeps the prediction mechanism consistent across comparisons.

Train prediction rules on training data. Choose the fixed and adaptive strategies
using validation error only. Use a fixed, documented tie-break rule. Evaluate the
selected strategies on test patients; do not change them based on the test results.

**Your handoff:** a results table, the selected strategy, and five example patient paths.


In [ ]:
# Learn majority-label predictions using a specified set of feature columns.
# Inputs:
#   z_train: the binarized training DataFrame.
#   y_train: the matching training labels.
#   columns: a list of feature names: radius only, or all four features.
# Outputs:
#   A model containing the chosen columns and a mapping from each possible
#   binary-value combination to a predicted label, including empty combinations.
def fit_lookup(z_train, y_train, columns):
    ...  # TODO: implement this function.


In [ ]:
# Predict labels using only the feature columns stored in a lookup model.
# Inputs:
#   model: a model returned by fit_lookup.
#   z: a binarized feature DataFrame.
# Outputs:
#   A 1D NumPy array of predicted 0/1 labels in the original row order.
def predict_lookup(model, z):
    ...  # TODO: implement this function.


In [ ]:
# Calculate the fraction of predictions that are wrong.
# Inputs:
#   truth: the actual 0/1 labels.
#   pred: the predicted 0/1 labels in matching order.
# Outputs:
#   One number between 0 and 1: wrong predictions divided by all predictions.
def error_rate(truth, pred):
    ...  # TODO: implement this function.


In [ ]:
# Summarize prediction errors, including missed malignant cases.
# Inputs:
#   truth: the actual labels (1 = malignant).
#   pred: the predicted labels in matching order.
# Outputs:
#   A dictionary containing error_rate, false_negative_rate, false_positive_rate,
#   wrong_predictions, false_negatives, n_test, and n_malignant_test.
#   False-negative rate = missed malignant cases / all actual malignant cases;
#   false-positive rate = benign cases predicted malignant / all actual benign cases.
#   If a denominator is zero, report NaN rather than silently using zero.
def metrics(truth, pred):
    ...  # TODO: implement this function.


In [ ]:
# Train, select, and evaluate the five approaches for one patient split.
# Inputs:
#   X: the original four-column feature DataFrame.
#   y: the aligned diagnosis labels.
#   seed: the random seed for reproducible splitting and random acquisition.
# Outputs:
#   A results DataFrame with one row per approach: seed, strategy, extra_features,
#   and the metrics above. Extra-feature counts are 0, 1, 1, 1, and 3 respectively.
#   Also return a details dictionary containing the split, selected model,
#   validation scores for candidate strategies, and a patient acquisition trace.
#   The trace records patient row ID, initial bin, chosen feature, acquired bin,
#   prediction, and actual label.
def run_split(X, y, seed):
    ...  # TODO: implement this function.


In [ ]:
# Summarize the prespecified repeated splits without treating them as independent studies.
# Inputs:
#   results: the combined results DataFrame from the five fixed seeds.
# Outputs:
#   A summary DataFrame grouped by strategy: mean error, error standard deviation,
#   mean false-negative rate, mean false-positive rate, and extra-feature count.
def summarize_runs(results):
    ...  # TODO: implement this function.


First complete one split, then run the five seeds your team fixed in advance.
The adaptive search includes the fixed strategies: it may select a fixed one.
Do not force different branch choices or expect an improvement.

The five splits overlap. Their standard deviation describes variation; it is not
a confidence interval. Label the all-four approach as a larger-budget reference.
Save the results table and the actual chosen rules so someone else can reproduce them.


# Teammate #4 — Create incomplete records and inspect coverage

This is preparation for the **next experiment**, not a missing-data model yet.

Create two measurement groups:

- **Panel A:** radius + texture.
- **Panel B:** smoothness + concavity.

Some training patients should retain only A, others only B, and a specified
fraction should retain both. Keep the diagnosis label for every patient.
Assign panels randomly, independently of diagnosis and feature values.

Compare requested overlap fractions of **0%, 10%, 50%, and 100%**.
Count how many patients have each pair of measurements recorded together.
At zero overlap, no patient should have a feature from A together with a feature from B.

**Your handoff:** masked training/validation tables, coverage tables, and an explanation
of which measurement combinations can no longer be estimated directly.


In [2]:
# Hide one measurement panel for some patients, leaving a chosen fraction with both.
# Inputs:
#   raw: a four-column training or validation DataFrame, without diagnosis labels.
#   overlap_fraction: a number from 0 to 1 specifying the fraction retaining both panels.
#   seed: an integer controlling reproducible panel assignment.
# Outputs:
#   A copy of raw with hidden feature values replaced by NaN.
#   Preserve the original values, columns, row IDs, and row order otherwise.
#   Round the requested complete-record count and randomly assign remaining rows A or B.
# def mask_panels(raw, overlap_fraction, seed):
# TODO: implement this function.
PANEL_A = ['mean radius', 'mean texture'] #size and shade of the cell nuclei
PANEL_B = ['mean smoothness', 'mean concavity'] #how smooth and dented the nucleus outline is

def mask_panels(raw, overlap_fraction, seed): 
    rng = np.random.default_rng(seed) #random number generator where the same seed gives the same random choices
    n = len(raw) #number of patients = number of rows in the table
    n_both = int(round(overlap_fraction * n)) #number of patients with both panels
    order = rng.permutation(n) #shuffles the row positions into a random order, making the choice random
    rest = order[n_both:] #the remaining patients after the ones with both panels
    half = len(rest) // 2 #half of the remaining patients
    only_a, only_b = rest[:half], rest[half:] #split the remaining patients into two groups
    masked = raw.copy() #make a copy of the original data to modify. without the copy, the hiding would damage the orginal table
    a_cols = masked.columns.get_indexer(PANEL_A) #positions of radius, texture
    b_cols = masked.columns.get_indexer(PANEL_B) #positions of smoothness, concavity
    masked.iloc[only_a, b_cols] = np.nan #hide panel B for patients in only A
    masked.iloc[only_b, a_cols] = np.nan #hide panel A for patients in only B
    return masked #return the modified table with some hidden values

In [4]:
# Count how many patients have each pair of features observed together.
# Inputs:
#   masked: a masked four-column feature DataFrame, without diagnosis labels.
# Outputs:
#   A square DataFrame with feature names as rows and columns.
#   Entry (A, B) counts rows with both A and B present;
#   diagonal entries count how often each individual feature is present.
def joint_coverage(masked):
# TODO: implement this function.
    seen = masked.notna().astype(int) #a cell is "seen" (1) if a value is recorded, and "missing" (0) if it's NaN. this turns the table into a map of what was actually observed
    return seen.T.dot(seen) #the dot product of the transpose of the seen table and the seen table itself counts how many patients have each pair of features observed together. the diagonal entries count how often each individual feature is present

In [ ]:
#testing the functions
FEATURES = ['mean radius', 'mean texture', 'mean smoothness', 'mean concavity']
X = load_breast_cancer(as_frame=True).data[FEATURES] 
#will run this test once the import line for the first section has been run and the FEATURES variable has been created

for frac in [0, 0.1, 0.5, 1]:
    print(f'Overlap {int(frac * 100)}%')
    display(joint_coverage(mask_panels(X, frac, seed=0)))

Make masks separately for training and validation patients. Keep the test data
complete for evaluation. A future learner should only receive the masked records;
the evaluator reveals test features along the selected measurement path.

Do not reuse the complete-data medians, fitted prediction rules, or selected strategies
when implementing the missing-data experiment. Those would reveal information that
the future learner is supposed to lack. Fit preprocessing from observed training values.

Scores from Teammate #3 are **complete-training-data baseline scores**, not evidence
that a method works on these masked records. Random masking is a controlled experiment,
not a complete simulation of why clinical tests are missing.

**Optional short exercise: connect this to regret.** This is a separate invented example,
not a WDBC result. Suppose two binary test results have four equally likely combinations.
Under one possible relationship, disease occurs when results differ; under the other,
it occurs when results match. Compare strategies predicting disease for differing results,
matching results, always benign, and always malignant. Build a table with one strategy
per row and one possible relationship per column. Each entry is its prediction error.


In [6]:
# Compare each strategy with the best strategy under the same possible relationship.
# Inputs:
#   error_table: a DataFrame of error rates: rows are strategies and columns are possible relationships.
# Outputs:
#   A regret table of the same shape, each strategy's largest regret across columns,
#   and the names of all strategies tied for the smallest largest regret.
#   Regret = strategy error minus the smallest strategy error in that same column.
def calculate_regret(error_table):
# TODO: implement this function.
    regret = error_table - error_table.min(axis=0) #subtract the smallest error in each column from each entry in that column to get the regret
    worst = regret.max(axis=1) #find the largest regret for each strategy
    best = worst[np.isclose(worst, worst.min())].index.tolist() #find the strategies with the smallest largest regret and return their names as a list
    return regret, worst, best #hand back the three outputs the comments lists

In [11]:
err = pd.DataFrame(
    {'differ': [0,1,0.5,0.5],
     'match': [1,0,0.5,0.5]},
     index = ['predict differ', 'predict match', 'always benign', 'always malignant']
)
regret, worst, best = calculate_regret(err)
display(regret)
print(worst)
print('Minimax-regret strategies:', best)

,differ,match
predict differ,0.0,1.0
predict match,1.0,0.0
always benign,0.5,0.5
always malignant,0.5,0.5


predict differ      1.0
predict match       1.0
always benign       0.5
always malignant    0.5
dtype: float64
Minimax-regret strategies: ['always benign', 'always malignant']


## Bring together before the meeting

- One notebook that runs from top to bottom after you fill in the missing code.
- One selected-tree diagram and five explained patient paths.
- A baseline comparison table and the seeds/settings used.
- Coverage tables showing what disappears as joint observations become scarce.

Each teammate should review another person's section. Everyone should be able to
explain one full prediction path. This assignment builds the baseline and missingness
setup; it does not yet implement the WDBC minimax-regret selection method.
